In [0]:
import smtplib
from email.mime.text import MIMEText
from pyspark.sql import functions as F

# Gmail credentials
sender_email = "jaggurasekar105@gmail.com"
sender_password = dbutils.secrets.get(scope="city_bank-scope", key="gmail_api_key")

# Read high-value transaction alerts from the gold table
# (this table already contains only transactions above the customer's configured limit)
alerts_df = spark.read.table("city_bank.gold.high_value_transactions_alert")

# One email per customer: list every transaction above their configured limit
customer_alerts = (
    alerts_df.groupBy(
        "customer_id", "customer_email", "customer_name", "transaction_limit", "currency"
    ).agg(
        F.collect_list(
            F.struct(
                F.col("transaction_id"),
                F.col("transaction_amount"),
                F.col("merchant_name"),
                F.col("transaction_type"),
                F.col("city"),
                F.col("country"),
                F.col("transaction_timestamp"),
                F.col("status"),
            )
        ).alias("transactions")
    )
)

sent, failed = 0, 0
for row in customer_alerts.collect():
    lines = [
        "Dear {},".format(row["customer_name"]),
        "",
        "The following transaction(s) exceeded your configured transaction limit of {:,.2f} {}:".format(
            row["transaction_limit"], row["currency"]
        ),
        "",
    ]
    for t in row["transactions"]:
        lines.append(
            "- Transaction {} | Amount: {:,.2f} {} | Merchant: {} | Type: {} | Location: {}, {} | Time: {} | Status: {}".format(
                t["transaction_id"],
                t["transaction_amount"],
                row["currency"],
                t["merchant_name"],
                t["transaction_type"],
                t["city"],
                t["country"],
                str(t["transaction_timestamp"]),
                t["status"],
            )
        )

    lines += [
        "",
        "If you did not make these transactions, please contact City Bank support immediately.",
        "",
        "Regards,",
        "City Bank Fraud Monitoring",
    ]

    message = MIMEText("\n".join(lines))
    message["Subject"] = "High Value Transaction Alert - Action Required"
    message["From"] = sender_email
    message["To"] = row["customer_email"]

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(sender_email, sender_password)
            server.send_message(message)
        sent += 1
        print("Alert email sent to {} ({} transactions)".format(row["customer_email"], len(row["transactions"])))
    except Exception as e:
        failed += 1
        print("Email to {} failed: {}".format(row["customer_email"], str(e)))

print("Done. Sent: {}, Failed: {}".format(sent, failed))